# Thunders AI — Model Training

This notebook demonstrates how to train models using the Thunders AI framework.
We'll cover dataset loading, training configuration, model training, evaluation, and visualization.

In [ ]:
# Install Thunders AI if needed
# !pip install thunders-ai[all]

import thunders_ai
from thunders_ai import Trainer, TrainingConfig, Dataset
from thunders_ai.models import get_model

print(f"Thunders AI version: {thunders_ai.__version__}")

## 1. Load Dataset

Thunders AI supports multiple dataset formats including HuggingFace datasets, custom CSV/JSON, and streaming data.

In [ ]:
# Load a dataset for training
dataset = Dataset.from_huggingface(
    "imdb",
    split="train",
    streaming=False
)

# Alternatively, load from local files
# dataset = Dataset.from_csv("data/train.csv", text_column="review", label_column="sentiment")
# dataset = Dataset.from_json("data/train.jsonl", text_key="text", label_key="label")

print(f"Dataset size: {len(dataset):,} samples")
print(f"Sample: {dataset[0]}")

# Split into train and validation
train_dataset, val_dataset = dataset.split(test_size=0.1, seed=42)
print(f"Train: {len(train_dataset):,} | Validation: {len(val_dataset):,}")

## 2. Configure Training

Set up the training configuration including hyperparameters, optimization, and checkpointing.

In [ ]:
# Configure training parameters
config = TrainingConfig(
    # Model
    model_name="thunders-ai/default",
    
    # Training
    epochs=5,
    batch_size=32,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=500,
    max_seq_length=512,
    gradient_accumulation_steps=1,
    
    # Precision
    fp16=True,
    
    # Checkpointing
    save_interval=500,
    eval_interval=250,
    output_dir="./checkpoints",
    
    # Logging
    log_interval=50,
    use_wandb=False,
)

print("Training Configuration:")
print(f"  Model: {config.model_name}")
print(f"  Epochs: {config.epochs}")
print(f"  Batch Size: {config.batch_size}")
print(f"  Learning Rate: {config.learning_rate}")
print(f"  FP16: {config.fp16}")

## 3. Run Training

Initialize the trainer and start the training loop.

In [ ]:
# Initialize the trainer
trainer = Trainer(
    config=config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
)

# Start training
results = trainer.train()

print("\nTraining Complete!")
print(f"Final train loss: {results.train_loss:.4f}")
print(f"Final val loss: {results.val_loss:.4f}")
print(f"Best model checkpoint: {results.best_checkpoint}")

## 4. Evaluate Model

Evaluate the trained model on the validation and test sets.

In [ ]:
# Load the best checkpoint for evaluation
model = get_model(results.best_checkpoint)

# Evaluate on validation set
val_metrics = trainer.evaluate(model, val_dataset)

print("Validation Metrics:")
for metric, value in val_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Evaluate on test set
test_dataset = Dataset.from_huggingface("imdb", split="test")
test_metrics = trainer.evaluate(model, test_dataset)

print("\nTest Metrics:")
for metric, value in test_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 5. Visualize Results

Plot training curves and confusion matrices.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot training and validation loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(results.history["train_loss"], label="Train Loss", linewidth=2)
axes[0].plot(results.history["val_loss"], label="Val Loss", linewidth=2)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
if "train_accuracy" in results.history:
    axes[1].plot(results.history["train_accuracy"], label="Train Accuracy", linewidth=2)
    axes[1].plot(results.history["val_accuracy"], label="Val Accuracy", linewidth=2)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Training & Validation Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("Training curves saved to training_curves.png")

## 6. Save & Export Model

Export the trained model for deployment.

In [ ]:
# Save model in different formats
model.save("./exports/model_thunders")
model.export_onnx("./exports/model.onnx")
model.export_huggingface("./exports/model_hf")

print("Model exported in Thunders AI, ONNX, and HuggingFace formats.")
print("Exports saved to ./exports/")